# EDA Gold Layer
Verifying the dimensional model built in build_gold.py.

## Dimensional model
- **dim_event**: 79,482 unique event instances
  - Composite key: event_name + event_dates + event_distance_raw
  - Same event can run multiple times per year on different dates
  - Same event can offer multiple distances on same date
- **dim_athlete**: 1,735,417 unique athletes
  - Identity columns only: athlete_id, athlete_id_raw, country, gender, year_of_birth
  - athlete_age_category and athlete_club moved to fct_results as they change per race
- **fct_results**: 7,043,305 results
  - One row per race result
  - performance_seconds for km/mi events
  - performance_km for h events
- **vw_distance_results**: km and mi events joined with dimensions
- **vw_timed_results**: h events joined with dimensions

## Known limitations
- athlete_id_raw shared between different athletes in some cases
- 12,875 athlete-event combinations have duplicate results with different birth years
- All duplicate rows kept as they represent legitimate race results


In [0]:
%run ../explorations/_setup

In [0]:

from importlib import reload
import transformations.gold.build_gold as gold_module
reload(gold_module)

In [0]:
from transformations.gold.build_gold import build_gold

build_gold(spark)

In [0]:
spark.sql("SELECT * FROM marathos.gold.vw_distance_results LIMIT 5").display()
spark.sql("SELECT * FROM marathos.gold.vw_timed_results LIMIT 5").display()

In [0]:
spark.sql("SELECT * FROM marathos.gold.vw_distance_results LIMIT 5").display()
spark.sql("SELECT * FROM marathos.gold.vw_timed_results LIMIT 5").display()

In [0]:
from transformations.gold.build_gold import get_table
dim_athlete = get_table("marathos.gold.dim_athlete")
dim_athlete.limit(10).display()

#check nulls
from pyspark.sql import functions as F
dim_athlete.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in dim_athlete.columns
]).display()

In [0]:
dim_event = get_table("marathos.gold.dim_event")
dim_event.groupBy("event_distance_unit").count().orderBy(F.desc("count")).display()

In [0]:
import pyspark.sql.functions as F
# check for duplicates in dim_event
dim_event.groupBy("event_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .display()

## seems the dim_event still has issues, need to check if the same event on the same event_date can have different lengths?


In [0]:
dim_event.groupBy("event_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .join(dim_event, "event_id") \
  .orderBy("event_id") \
  .limit(10) \
  .display()

In [0]:
dim_event   = get_table("marathos.gold.dim_event")
dim_athlete = get_table("marathos.gold.dim_athlete")
fct_results = get_table("marathos.gold.fct_results")

print(f"dim_event rows:   {dim_event.count()}")
print(f"dim_athlete rows: {dim_athlete.count()}")
print(f"fct_results rows: {fct_results.count()}")

In [0]:
# No duplicates in dim_event
dim_event.groupBy("event_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .count()

In [0]:
# No duplicates in dim_athlete
dim_athlete.groupBy("athlete_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .count()

In [0]:
fct_results = get_table("marathos.gold.fct_results")

fct_results.groupBy("event_id", "athlete_id") \
  .count() \
  .filter(F.col("count") > 1) \
  .orderBy(F.desc("count")) \
  .limit(10) \
  .display()